In [1]:
import pandas as pd

df= pd.read_csv('RetailCleaned.csv', header=None)
df.columns=['Invoice','StockCode','Description','Quantity','InvoiceDate','Price','Customer_ID','Country','SourceSheet']

df['InvoiceDate']=pd.to_datetime(df['InvoiceDate'])
print(df.shape)
df.info()

C:\Users\BIT\AppData\Local\Temp\ipykernel_4656\864151079.py:3: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df= pd.read_csv('RetailCleaned.csv', header=None)


(1033036, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1033036 entries, 0 to 1033035
Data columns (total 9 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1033036 non-null  object        
 1   StockCode    1033036 non-null  object        
 2   Description  1028761 non-null  object        
 3   Quantity     1033036 non-null  int64         
 4   InvoiceDate  1033036 non-null  datetime64[ns]
 5   Price        1033036 non-null  float64       
 6   Customer_ID  797885 non-null   float64       
 7   Country      1033036 non-null  object        
 8   SourceSheet  1033036 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(5)
memory usage: 70.9+ MB


In [2]:
# RFM(recency,frequency,Monetary)
# Recency:-Recency is the number of days the product was last purchased(suppose I bought a product 1 week ago then recency will be 7)
# Frequency:-How many times the product has been purchased
# Monetary:- The revenue generated when we bought the products

rfm_data=df[(df['Customer_ID'].notna())&(df['Quantity']>0)].copy()

snapshot_date=pd.to_datetime('2011-12-10')
rfm=rfm_data.groupby('Customer_ID').agg(
    Recency=('InvoiceDate',lambda x:(snapshot_date-x.max()).days),
    Frequency=('Invoice','nunique'),# For each customer_ID how much invoice/orders(A1,A2,A3 etc...) number they have
    Monetary=('Quantity',lambda x:(x * rfm_data.loc[x.index,'Price']).sum())
).reset_index()
    

In [3]:
rfm.head()

,Customer_ID,Recency,Frequency,Monetary
0,12346.0,325,12,77556.46
1,12347.0,2,8,4921.53
2,12348.0,75,5,2019.40
3,12349.0,18,4,4428.69
4,12350.0,310,1,334.40


In [5]:
# Score customers and assign business segments

rfm['R_Score']=pd.qcut(rfm['Recency'],5,labels=[5,4,3,2,1]).astype(int)
rfm['F_Score']=pd.qcut(rfm['Frequency'].rank(method='first'),5,labels=[1,2,3,4,5]).astype(int)
rfm['M_Score']=pd.qcut(rfm['Monetary'],5,labels=[1,2,3,4,5]).astype(int)

def segment_customer(row):
    if row['R_Score'] >= 4 and row['F_Score'] >= 4 and row['M_Score'] >=4:
        return 'Champions'
    elif row['R_Score'] >= 3 and row['F_Score'] >= 3:
        return 'Loyal'
    elif row['R_Score'] <= 2 and row['F_Score'] >= 3:
        return 'At Risk'
    elif row['R_Score'] >=4 and row['F_Score'] <= 2:
        return 'New'
    elif row['R_Score'] == 3 and row['F_Score'] <= 2:
        return 'Needs Attention'
    else:
        return 'lost'

rfm['Segment']=rfm.apply(segment_customer,axis=1)
rfm['IsChurned']=rfm['Recency'] > 90

In [7]:
## Merge RFM results back onto transanctions and export
df=df[(df['Customer_ID'].notna()) & (df['Quantity']>0)].copy()
df['InvoiceMonth']=df['InvoiceDate'].dt.to_period('M').dt.to_timestamp()
df['Revenue']=df['Quantity']*df['Price']

df_final=df.merge(
    rfm[['Customer_ID','Recency','Frequency','Monetary','R_Score','F_Score','M_Score','Segment','IsChurned']],
    on='Customer_ID',how='left'
)
df_final.to_excel('EcommerceDashBoardData.xlsx',index=False)

In [8]:
import pandas as pd

df = pd.read_excel('EcommerceDashBoardData.xlsx')
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer_ID,Country,SourceSheet,InvoiceMonth,Revenue,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,Segment,IsChurned
0,490151,17107D,"FLOWER FAIRY,5 SUMMER B'DRAW LINERS",6,2009-12-04 09:49:00,2.55,16645,United Kingdom,2010-2011,2009-12-01,15.30,735,1,160.39,1,1,1,lost,True
1,490151,17108D,FLOWER FAIRY SUMMER BOUQUET SACHET,12,2009-12-04 09:49:00,0.42,16645,United Kingdom,2010-2011,2009-12-01,5.04,735,1,160.39,1,1,1,lost,True
2,490151,17109B,FLOWER FAIRY INCENSE LAVENDER,6,2009-12-04 09:49:00,1.69,16645,United Kingdom,2010-2011,2009-12-01,10.14,735,1,160.39,1,1,1,lost,True
3,490151,17109D,FLOWER FAIRY INCENSE BOUQUET,6,2009-12-04 09:49:00,1.69,16645,United Kingdom,2010-2011,2009-12-01,10.14,735,1,160.39,1,1,1,lost,True
4,490151,21185,HOME SWEET HOME BLACKBOARD,3,2009-12-04 09:49:00,6.35,16645,United Kingdom,2010-2011,2009-12-01,19.05,735,1,160.39,1,1,1,lost,True
